# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the [FAIR^2 colorectal cancer survivors dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library, referencing all fields and elements via their `@id`.

### Dataset Source
The dataset is described by a Croissant schema available at:
[https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant JSON-LD schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset (this loads metadata and structure)
dataset = mlc.Dataset(url)

# Access and display high-level metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Identifier: {meta.identifier if hasattr(meta, 'identifier') else ''}")
if hasattr(meta, 'keywords'):
    print(f"Keywords: {meta.keywords}")
print(f"License: {meta.license}")
print(f"Published: {meta.datePublished if hasattr(meta, 'datePublished') else ''}")

## 2. Data Overview

Review the available record sets within the dataset and inspect their fields and columns, referencing each by its `@id`.

We'll enumerate all record sets in the schema, with their unique Croissant `@id`, and display the available fields (by `@id`) for the main data table. This helps select the appropriate data structures for further processing.

In [ ]:
# Access all record sets defined in the dataset
record_sets = dataset.record_sets
print(f"Record sets in dataset (referenced by @id):")
for rs in record_sets:
    print(f"  - {rs['@id']}: {rs['name']} (type: {rs.get('@type', 'RecordSet')})")

# For this dataset there is likely a main record set containing the records
main_rs = None
for rs in record_sets:
    # Heuristic: main data table will usually have a name or id that matches data content
    if 'Clinicopathological' in rs.get('name', '') or 'colorectal' in rs.get('name', '').lower():
        main_rs = rs
if not main_rs:
    main_rs = record_sets[0] if len(record_sets) > 0 else None
assert main_rs is not None, "No record set found in this dataset."
main_rs_id = main_rs['@id']
print(f"\nMain record set selected: {main_rs_id} ({main_rs['name']})\n")

# List fields (by @id) in the main record set
print(f"Fields in the main record set (by @id):")
for field in main_rs['field']:
    print(f"  - {field['@id']}: {field['name']} (type: {field.get('dataType', '')})")

## 3. Data Extraction

Load all data from the main record set (`@id`) into a pandas DataFrame using the Croissant `mlcroissant` interface.
All field and record set references use their `@id`.

In [ ]:
# List of all record set @ids
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")

# Show columns of the main record set DataFrame
if main_rs_id in dataframes:
    print(f"\nColumns in DataFrame for main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print(f"No records loaded for main record set {main_rs_id}")

## 4. Exploratory Data Analysis (EDA)

Here, we apply common data processing steps using only `@id` references, such as filtering records by clinical variables, normalizing numeric fields, and grouping data. Operations include outlier detection/removal, transformations, and grouping, to prepare the data for downstream analysis.

First, we identify suitable numeric and categorical fields within the loaded DataFrame using their schema `@id`.

In [ ]:
# For reproducibility, we'll identify a numeric field from fields list
main_fields = main_rs['field']
numeric_field_id = None
group_field_id = None
# Try common numeric/clinical variable names
for f in main_fields:
    n = f['name'].lower()
    if (('age' in n or 'interval' in n or 'tumor' in n or 'metastasis' in n) and f.get('dataType', '').lower() in ['integer', 'float', 'number']):
        numeric_field_id = f['@id']
        break
# Select a fallback numeric field if above fail
if not numeric_field_id:
    for f in main_fields:
        if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
            numeric_field_id = f['@id']
            break

# Identify a grouping/categorical field (e.g. sex, anatomical location)
for f in main_fields:
    n = f['name'].lower()
    if any(q in n for q in ['sex', 'anatomical', 'site', 'group', 'status', 'location', 'msi']):
        group_field_id = f['@id']
        break
# Fallback: first non-numeric field as group
if not group_field_id:
    for f in main_fields:
        if f.get('dataType', '').lower() not in ['integer', 'float', 'number']:
            group_field_id = f['@id']
            break

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group/categorical field selected (@id): {group_field_id}")

df = dataframes[main_rs_id].copy()
if numeric_field_id not in df.columns:
    raise ValueError(f"Selected numeric field {numeric_field_id} not present in DataFrame.")

# Convert numeric field to float (if not already)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: numeric_field > threshold (use mean or a clinical value as threshold)
thresh = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
filtered_df = df[df[numeric_field_id] > thresh]
print(f"Filtered records where {numeric_field_id} > {thresh:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field for these records
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' (column '{norm_col}'):")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by the group_field (if present)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
    print(f"\nGrouped statistics by '{group_field_id}':")
    display(grouped_df)
else:
    print(f"Group field {group_field_id} not present in DataFrame columns.")

## 5. Visualization

Visualize the distribution of the selected numeric variable and its relationship to the grouped/categorical variable, referencing columns by their `@id`.

We'll provide histogram and bar plots for the selected variables.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field in filtered data
plt.figure(figsize=(8,5))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=10, kde=True)
plt.title(f"Histogram of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Barplot: mean of numeric field for each group
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=filtered_df, ci='sd')
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load, inspect, and analyze the *Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors* dataset—using only Croissant `@id` references at every step.

- We loaded the Croissant schema and record sets dynamically using the `mlcroissant` library.
- Identified and referenced all major data objects, fields, and columns using their `@id`.
- Extracted tabular data for the primary record set and performed basic EDA—including filtering, normalization, grouping, and visualization—by referencing variables only via their `@id` (not field name).
- This approach facilitates robust, reproducible, and FAIR data science workflows leveraging modern scientific data standards.

**Next steps:** You can modify field selection (by `@id`) and extend the analysis for deeper clinical or molecular hypothesis testing, all without hard-coding human-friendly column names.